In [1]:
from pathlib import Path

import duckdb
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DATA_DIR = PROJECT_ROOT / "data" / "output"

for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    OUTPUT_DATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SOLAR_CSV = RAW_DATA_DIR / "solar_generation.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {SOLAR_CSV}")

Project root: /Users/dev/Developer
Dataset path: /Users/dev/Developer/data/raw/solar_generation.csv


In [3]:
solar_data = pd.DataFrame(
    {
        "timestamp": pd.to_datetime(
            [
                "2026-07-01 06:00",
                "2026-07-01 08:00",
                "2026-07-01 10:00",
                "2026-07-01 12:00",
                "2026-07-01 14:00",
                "2026-07-01 16:00",
                "2026-07-01 18:00",
                "2026-07-02 06:00",
                "2026-07-02 08:00",
                "2026-07-02 10:00",
                "2026-07-02 12:00",
                "2026-07-02 14:00",
                "2026-07-02 16:00",
                "2026-07-02 18:00",
            ]
        ),
        "site_id": ["SITE-001"] * 14,
        "irradiance_w_m2": [
            80,
            310,
            620,
            900,
            780,
            460,
            120,
            60,
            240,
            500,
            690,
            610,
            350,
            90,
        ],
        "temperature_c": [
            16,
            19,
            23,
            28,
            31,
            29,
            24,
            17,
            20,
            24,
            27,
            29,
            27,
            22,
        ],
        "power_kw": [
            0.8,
            3.1,
            6.4,
            9.0,
            7.6,
            4.3,
            1.0,
            0.5,
            2.2,
            4.9,
            6.8,
            5.8,
            3.2,
            0.7,
        ],
    }
)

solar_data.to_csv(SOLAR_CSV, index=False)

solar_data

,timestamp,site_id,irradiance_w_m2,temperature_c,power_kw
0,2026-07-01 06:00:00,SITE-001,80,16,0.8
1,2026-07-01 08:00:00,SITE-001,310,19,3.1
2,2026-07-01 10:00:00,SITE-001,620,23,6.4
3,2026-07-01 12:00:00,SITE-001,900,28,9.0
4,2026-07-01 14:00:00,SITE-001,780,31,7.6
5,2026-07-01 16:00:00,SITE-001,460,29,4.3
6,2026-07-01 18:00:00,SITE-001,120,24,1.0
7,2026-07-02 06:00:00,SITE-001,60,17,0.5
8,2026-07-02 08:00:00,SITE-001,240,20,2.2
9,2026-07-02 10:00:00,SITE-001,500,24,4.9


In [4]:
duckdb.sql(
    f"""
    SELECT *
    FROM read_csv_auto('{SOLAR_CSV}')
    ORDER BY timestamp
    """
).df()

,timestamp,site_id,irradiance_w_m2,temperature_c,power_kw
0,2026-07-01 06:00:00,SITE-001,80,16,0.8
1,2026-07-01 08:00:00,SITE-001,310,19,3.1
2,2026-07-01 10:00:00,SITE-001,620,23,6.4
3,2026-07-01 12:00:00,SITE-001,900,28,9.0
4,2026-07-01 14:00:00,SITE-001,780,31,7.6
5,2026-07-01 16:00:00,SITE-001,460,29,4.3
6,2026-07-01 18:00:00,SITE-001,120,24,1.0
7,2026-07-02 06:00:00,SITE-001,60,17,0.5
8,2026-07-02 08:00:00,SITE-001,240,20,2.2
9,2026-07-02 10:00:00,SITE-001,500,24,4.9


In [5]:
daily_summary = duckdb.sql(
    f"""
    SELECT
        CAST(timestamp AS DATE) AS generation_date,
        ROUND(SUM(power_kw), 2) AS total_observed_power_kw,
        ROUND(AVG(power_kw), 2) AS average_power_kw,
        ROUND(MAX(power_kw), 2) AS peak_power_kw,
        ROUND(AVG(irradiance_w_m2), 2) AS average_irradiance,
        ROUND(AVG(temperature_c), 2) AS average_temperature_c
    FROM read_csv_auto('{SOLAR_CSV}')
    GROUP BY generation_date
    ORDER BY generation_date
    """
).df()

daily_summary

,generation_date,total_observed_power_kw,average_power_kw,peak_power_kw,average_irradiance,average_temperature_c
0,2026-07-01,32.2,4.60,9.0,467.14,24.29
1,2026-07-02,24.1,3.44,6.8,362.86,23.71


In [6]:
energy_summary = duckdb.sql(
    f"""
    SELECT
        CAST(timestamp AS DATE) AS generation_date,
        ROUND(SUM(power_kw * 2), 2) AS estimated_energy_kwh
    FROM read_csv_auto('{SOLAR_CSV}')
    GROUP BY generation_date
    ORDER BY generation_date
    """
).df()

energy_summary

,generation_date,estimated_energy_kwh
0,2026-07-01,64.4
1,2026-07-02,48.2


In [7]:
duckdb.sql(
    f"""
    SELECT
        timestamp,
        site_id,
        irradiance_w_m2,
        temperature_c,
        power_kw
    FROM read_csv_auto('{SOLAR_CSV}')
    WHERE power_kw >= 6.0
    ORDER BY power_kw DESC
    """
).df()

,timestamp,site_id,irradiance_w_m2,temperature_c,power_kw
0,2026-07-01 12:00:00,SITE-001,900,28,9.0
1,2026-07-01 14:00:00,SITE-001,780,31,7.6
2,2026-07-02 12:00:00,SITE-001,690,27,6.8
3,2026-07-01 10:00:00,SITE-001,620,23,6.4


In [8]:
PARQUET_PATH = PROCESSED_DATA_DIR / "solar_generation.parquet"

duckdb.sql(
    f"""
    COPY (
        SELECT
            CAST(timestamp AS TIMESTAMP) AS timestamp,
            site_id,
            irradiance_w_m2,
            temperature_c,
            power_kw
        FROM read_csv_auto('{SOLAR_CSV}')
    )
    TO '{PARQUET_PATH}'
    (FORMAT PARQUET)
    """
)

print(f"Created: {PARQUET_PATH}")

Created: /Users/dev/Developer/data/processed/solar_generation.parquet


In [9]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("solar-analytics-project")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/27 11:41:13 WARN Utils: Your hostname, Roberts-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.92 instead (on interface en0)
26/07/27 11:41:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/27 11:41:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


In [10]:
solar_spark = spark.read.parquet(str(PARQUET_PATH))

solar_spark.printSchema()
solar_spark.show(truncate=False)

root
 |-- timestamp: timestamp_ntz (nullable = true)
 |-- site_id: string (nullable = true)
 |-- irradiance_w_m2: long (nullable = true)
 |-- temperature_c: long (nullable = true)
 |-- power_kw: double (nullable = true)

+-------------------+--------+---------------+-------------+--------+
|timestamp          |site_id |irradiance_w_m2|temperature_c|power_kw|
+-------------------+--------+---------------+-------------+--------+
|2026-07-01 06:00:00|SITE-001|80             |16           |0.8     |
|2026-07-01 08:00:00|SITE-001|310            |19           |3.1     |
|2026-07-01 10:00:00|SITE-001|620            |23           |6.4     |
|2026-07-01 12:00:00|SITE-001|900            |28           |9.0     |
|2026-07-01 14:00:00|SITE-001|780            |31           |7.6     |
|2026-07-01 16:00:00|SITE-001|460            |29           |4.3     |
|2026-07-01 18:00:00|SITE-001|120            |24           |1.0     |
|2026-07-02 06:00:00|SITE-001|60             |17           |0.5     |
|2026-07-

In [11]:
spark_daily_summary = (
    solar_spark
    .withColumn(
        "generation_date",
        F.to_date("timestamp"),
    )
    .groupBy("generation_date")
    .agg(
        F.round(F.sum(F.col("power_kw") * 2), 2)
        .alias("estimated_energy_kwh"),

        F.round(F.avg("power_kw"), 2)
        .alias("average_power_kw"),

        F.round(F.max("power_kw"), 2)
        .alias("peak_power_kw"),

        F.round(F.avg("irradiance_w_m2"), 2)
        .alias("average_irradiance"),
    )
    .orderBy("generation_date")
)

spark_daily_summary.show()

+---------------+--------------------+----------------+-------------+------------------+
|generation_date|estimated_energy_kwh|average_power_kw|peak_power_kw|average_irradiance|
+---------------+--------------------+----------------+-------------+------------------+
|     2026-07-01|                64.4|             4.6|          9.0|            467.14|
|     2026-07-02|                48.2|            3.44|          6.8|            362.86|
+---------------+--------------------+----------------+-------------+------------------+



In [12]:
duckdb_result = duckdb.sql(
    f"""
    SELECT
        CAST(timestamp AS DATE) AS generation_date,
        ROUND(SUM(power_kw * 2), 2) AS estimated_energy_kwh
    FROM read_parquet('{PARQUET_PATH}')
    GROUP BY generation_date
    ORDER BY generation_date
    """
).df()

spark_result = (
    spark_daily_summary
    .select(
        "generation_date",
        "estimated_energy_kwh",
    )
    .toPandas()
)

print("DuckDB result")
display(duckdb_result)

print("Spark result")
display(spark_result)

DuckDB result


,generation_date,estimated_energy_kwh
0,2026-07-01,64.4
1,2026-07-02,48.2


Spark result


,generation_date,estimated_energy_kwh
0,2026-07-01,64.4
1,2026-07-02,48.2


In [13]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
